# f696 五输出配置坐标与实际 beta 的区间对照

只读封存 Qlib Recorder；先证明五输出的 f696 身份与原账户逐日一致，再解释区间上的模型输出与资金动作。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R

root=Path.cwd();card=root/"research/stage1_f696_output_coordinate_interval_20261024"
plan_path=card/"plan.json";plan=json.loads(plan_path.read_text());src=plan["sources"]
assert plan["frozen"] and all(v==0 for v in plan["budget"].values())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
a=root/src["f696_card"];b=root/src["account_card"]
assert sha(a/"summary.json")==src["f696_summary_sha256"] and sha(a/"readback_verification.json")==src["f696_readback_sha256"]
assert sha(b/"summary.json")==src["account_summary_sha256"]
sa=json.loads((a/"summary.json").read_text());ra=json.loads((a/"readback_verification.json").read_text());sb=json.loads((b/"summary.json").read_text())
assert sa["nas_evidence"]["sha256"]==src["f696_nas_sha256"] and sa["nas_evidence"]["size_bytes"]==src["f696_nas_size_bytes"]
assert ra["archive_sha256"]==src["f696_nas_sha256"] and ra["status"]=="passed" and ra["all_artifact_hashes_match"]
assert sb["nas_evidence"]["sha256"]==src["account_nas_sha256"] and sb["nas_evidence"]["size_bytes"]==src["account_nas_size_bytes"]
assert sb["nas_readback"]["verified"] and sb["nas_readback"]["sha256"]==src["account_nas_sha256"]
aroot=a/"qlib_recorder"/src["f696_experiment_id"]/src["f696_recorder_id"]/"artifacts"
broot=b/"qlib_recorder"/src["account_experiment_id"]/src["account_recorder_id"]/"artifacts"
assert sha(aroot/"inputs/frame")==src["f696_frame_sha256"]
assert sha(broot/"analysis/daily")==src["account_daily_sha256"]
assert sha(broot/"analysis/periods")==src["account_periods_sha256"]
assert sha(broot/"analysis/rolling126")==src["account_rolling126_sha256"]
assert sha(root/src["qlib_provider"]/"qlib_store_manifest.json")==src["qlib_provider_manifest_sha256"]
qlib.init(provider_uri=str((root/src["qlib_provider"]).resolve()),region="us",expression_cache=None,dataset_cache=None)
with R.uri_context(str(a/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=src["f696_experiment_id"],recorder_id=src["f696_recorder_id"])
 frame=rec.load_object("inputs/frame").sort_values("trade_date").reset_index(drop=True)
with R.uri_context(str(b/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=src["account_experiment_id"],recorder_id=src["account_recorder_id"])
 daily=rec.load_object("analysis/daily").sort_values("trade_date").reset_index(drop=True)
 periods=rec.load_object("analysis/periods")
 rolling=rec.load_object("analysis/rolling126")
assert len(frame)==len(daily)==944 and len(rolling)==819
assert len(periods.loc[periods.kind.eq("quarter")])==16
assert periods.loc[periods.kind.eq("year") & periods.partial_boundary_period,"period"].astype(str).tolist()==["2023","2026"]
assert pd.DatetimeIndex(frame.signal_date).equals(pd.DatetimeIndex(daily.signal_date))
assert pd.DatetimeIndex(frame.trade_date).equals(pd.DatetimeIndex(daily.trade_date))
outputs=plan["fixed_definitions"]["proposal_coordinates"]
assert frame[outputs].notna().all().all()
assert np.isfinite(frame[outputs].to_numpy(dtype=float)).all()
weights=frame[["soxl_weight","soxx_weight","cash_weight"]]
assert float(weights.min().min())>=-1e-9 and float((weights.sum(axis=1)-1).abs().max())<1e-7
raw=3*frame.soxl_weight.astype(float)+frame.soxx_weight.astype(float)
parity={"alpha_probability_max_abs":float((frame.alpha_probability-daily.alpha_probability).abs().max()),
        "tail_probability_max_abs":float((frame.tail_probability-daily.tail_probability).abs().max()),
        "raw_beta_max_abs":float((raw-daily.raw_nominal_beta).abs().max())}
assert max(parity.values())<1e-8,parity
risky=frame.soxl_weight+frame.soxx_weight
share=(frame.soxl_weight/risky).where(risky>1e-10)
assert share.notna().all()
calendar=pd.DatetimeIndex(D.calendar(start_time="2011-01-03",end_time="2026-10-09",freq="day"))
close=D.features(["SOXX"],["$close"],start_time="2011-01-03",end_time="2026-10-09",freq="day")["$close"].droplevel("instrument").sort_index()
assert close.index.equals(calendar) and close.notna().all()
values=pd.DataFrame({"soxl_weight":frame.soxl_weight.astype(float),"soxx_weight":frame.soxx_weight.astype(float),
                     "cash_weight":frame.cash_weight.astype(float),"alpha_probability":frame.alpha_probability.astype(float),
                     "tail_probability":frame.tail_probability.astype(float),"raw_beta":raw.astype(float),
                     "risky_soxl_share":share.astype(float),"own_state_beta_cap":daily.own_state_beta_cap.astype(float),
                     "ending_beta":daily.ending_beta.astype(float),"cap_binding":daily.cap_binding.astype(float)})
measures=list(values.columns)
windows=[]
for i in range(819):
 segment=daily.iloc[i:i+126];v=values.iloc[i:i+126];rw=rolling.iloc[i]
 assert len(segment)==126 and pd.Timestamp(segment.trade_date.iloc[0])==pd.Timestamp(rw.first_trade)
 assert abs(float(segment.paired_log_excess.sum())-float(rw.relative_log_excess))<1e-10
 signal=pd.Timestamp(segment.signal_date.iloc[0]);loc=calendar.get_loc(signal);assert loc>=20
 r5=float(close.iloc[loc]/close.iloc[loc-5]-1);r20=float(close.iloc[loc]/close.iloc[loc-20]-1)
 state="uptrend" if r5>=0 and r20>=0 else "rebound" if r5>=0 else "pullback" if r20>=0 else "decline"
 row={"year":int(pd.Timestamp(rw.first_trade).year),"start_state":state,"outcome":"weak" if rw.relative_log_excess<0 else "good" if rw.relative_log_excess>0 else "tie"}
 row.update({m:float(v[m].mean()) for m in measures})
 windows.append(row)
w=pd.DataFrame(windows)
assert w.outcome.value_counts().to_dict()=={"weak":564,"good":255}
cells=[]
for (state,year),g in w.groupby(["start_state","year"],sort=True):
 weak=g[g.outcome.eq("weak")];good=g[g.outcome.eq("good")]
 c={"start_state":state,"year":int(year),"weak_windows":len(weak),"good_windows":len(good),"matched":bool(len(weak) and len(good))}
 if c["matched"]:c["weak_minus_good"]={m:float(weak[m].mean()-good[m].mean()) for m in measures}
 cells.append(c)
matched=[c for c in cells if c["matched"]]
assert len(matched)==12 and all(c["good_windows"]==0 for c in cells if c["year"]==2023)
cell_weights=np.array([min(c["weak_windows"],c["good_windows"]) for c in matched],dtype=float)
weighted={m:float(np.average([c["weak_minus_good"][m] for c in matched],weights=cell_weights)) for m in measures}
signs={m:{"positive_cells":sum(c["weak_minus_good"][m]>0 for c in matched),"negative_cells":sum(c["weak_minus_good"][m]<0 for c in matched)} for m in measures}
result={"status":"read_only_exact_f696_proposal_coordinate_interval_diagnostic_no_policy",
        "plan_sha256":sha(plan_path),"f696_recorder_id":src["f696_recorder_id"],"account_recorder_id":src["account_recorder_id"],
        "f696_five_output_parity":parity,"sessions":944,"rolling126_windows":819,"weak_windows":564,"good_windows":255,
        "matched_state_year_cells":len(matched),"weighted_weak_minus_good":weighted,"cell_signs":signs,"cells":cells,
        "new_market_requests":0,"new_model_fits":0,"new_native_accounts":0,"new_bootstraps":0,"new_recorders":0,
        "candidate_feature_selected":False,"policy_inference":False,"independent_validation":False,
        "limits":["Weak/good labels use future returns and 126-session windows overlap","2023 has no good within-year matched cells",
                  "Risk cap and funded beta are endogenous account states","Proposal-coordinate contrast does not identify a causal factor or tradable switch"]}
(card/"result.json").write_text(json.dumps(result,ensure_ascii=False,indent=2)+"\n")
print({"parity":parity,"weighted":weighted,"signs":signs})


2026/10/24 16:14:18 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[56869:MainThread](2026-10-24 16:14:18,901) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[56869:MainThread](2026-10-24 16:14:18,903) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[56869:MainThread](2026-10-24 16:14:18,903) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


{'parity': {'alpha_probability_max_abs': 0.0, 'tail_probability_max_abs': 0.0, 'raw_beta_max_abs': 0.0}, 'weighted': {'soxl_weight': -0.012458580034192368, 'soxx_weight': -0.015077580906240555, 'cash_weight': 0.027536160940432933, 'alpha_probability': 0.012353006196356502, 'tail_probability': -0.016203701403636996, 'raw_beta': -0.05245332100881761, 'risky_soxl_share': -0.00884801816073383, 'own_state_beta_cap': -0.07165289699778116, 'ending_beta': -0.07487846236072836, 'cap_binding': 0.040895121882505735}, 'signs': {'soxl_weight': {'positive_cells': 3, 'negative_cells': 9}, 'soxx_weight': {'positive_cells': 4, 'negative_cells': 8}, 'cash_weight': {'positive_cells': 9, 'negative_cells': 3}, 'alpha_probability': {'positive_cells': 11, 'negative_cells': 1}, 'tail_probability': {'positive_cells': 3, 'negative_cells': 9}, 'raw_beta': {'positive_cells': 3, 'negative_cells': 9}, 'risky_soxl_share': {'positive_cells': 3, 'negative_cells': 9}, 'own_state_beta_cap': {'positive_cells': 4, 'negati

## 中文结论

只读精确 f696 原生 Qlib Recorder 的 944 行 `inputs/frame`，与原 0bp Tiered 账户逐日对齐。SOXL、SOXX、现金、alpha、tail 五个保存输出及 `3×SOXL权重＋SOXX权重` 对原账户 raw beta 的最大绝对误差均为 0；这排除了 27baa 提案误用。保留全部819个连续126日窗口、16季度及两端不完整年度；同一事前 SOXX 5/20 日趋势符号和首交易年内有12个可配对好坏格子，2023四格没有好窗口。

在这些可配对格子中，坏减好按每格较小组窗口数加权的描述差：alpha 概率 **+0.01235**，tail 概率 **-0.01620**，原始现金提案 **+0.02754**，SOXL 提案 **-0.01246**，SOXX 提案 **-0.01508**，原始名义 beta **-0.05245**，自身风险上限 **-0.07165**，最终 beta **-0.07488**。坏窗口现金提案较高的格子有9/12，SOXL 提案较低9/12、SOXX提案较低8/12。alpha较高并未意味着更高的风险配置；差异已经出现在原始三资产提案中；坏窗口的自身风险上限均值同时更低，最终 beta 也更低，不能把两者差额当作可独立回收的收益。

这是模型输出和账户动作的描述性**双层错位**：不能仅松绑 cap，也不能把 alpha 增强或 tail 反号当作修复。好坏由未来收益定义，126日窗口重叠，风险上限属于内生账户状态；上述差异不证明某个输入因子的因果作用、可交易阈值或 SOXX 超额。原现金/tail 排序、cap-gap 及局部风险释放固定版本仍关闭。下一个因子必须有新的事前信息，解释何时保留现金保护、何时减少现金拖累，同时能通过风险投影改变资金动作；先固定匹配信息门，再跑自身完整0bp含费账户。

首次执行在区间比较前因把 SOXL 名义倍数误写为2而停止，观测到 raw beta 最大误差0.31537。原计划字节、失败和仅改为已有账户使用的3倍恒等式均单独保存；没有调整来源、区间、成本或科学门。结论 `descriptive_exact_f696_cash_proposal_and_risk_cap_dual_delivery_gap_no_candidate`，SOXX v2目标未达成。
